In [13]:
# -*- coding: utf-8 -*-
import numpy as np
import csv
import json
from pathlib import Path
from scipy.stats import truncnorm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import GradientBoostingClassifier

from LdaBoosting import LdaBoost  # mantiene i tuning del primo codice


class PipelineConfronto:
    """
    Esegue simulazioni su dataset gaussiani multiclasse con covarianza data,
    confrontando tre pipeline:
      1) LdaBoost (diretto)
      2) GradientBoostingClassifier su raw
      3) LDA -> GradientBoostingClassifier

    - Matrice di correlazione/covarianza generata con funzioni PSD-safe.
    - Tuning come nel primo codice (n_estimators=500, learning_rate=0.05, max_depth=7, random_state=1).
    - Salvataggi in 'output_lda_boost' con nomi tipo:
        pipeline_confront_summary_binary_1k.csv/json
        pipeline_confront_raw_binary_1k.json
    """

    # ---------------------- utilità di naming ----------------------
    _CLASS_NAME_MAP = {
        2: "binary", 3: "ternary", 4: "quaternary", 5: "quinary",
        6: "senary", 7: "septenary", 8: "octonary", 9: "nonary", 10: "denary"
    }

    @staticmethod
    def _class_label(k: int) -> str:
        return PipelineConfronto._CLASS_NAME_MAP.get(k, f"{k}class")

    @staticmethod
    def _samples_label(n: int) -> str:
        return f"{n//1000}k" if n % 1000 == 0 else str(n)

    # ----------------- funzioni PSD-safe per la cov ----------------
    @staticmethod
    def make_corr_well_conditioned(M: np.ndarray, min_eig: float = 1e-2, kappa_max: float = 1e3) -> np.ndarray:
        A = (M + M.T) / 2.0
        np.fill_diagonal(A, 1.0)

        w = np.linalg.eigvalsh(A)
        lam_min = float(w.min())
        lam_max = float(w.max())

        alpha_psd = 0.0
        if lam_min < min_eig:
            alpha_psd = (min_eig - lam_min) / (1.0 - lam_min)

        if kappa_max is None:
            alpha_kappa = 0.0
        else:
            Acoef = 1.0 - lam_max - kappa_max * (1.0 - lam_min)
            Bcoef = kappa_max * lam_min - lam_max
            alpha_kappa = float(np.clip(Bcoef / Acoef, 0.0, 0.999))

        alpha = float(np.clip(max(alpha_psd, alpha_kappa, 0.0), 0.0, 0.999))
        R = (1.0 - alpha) * A + alpha * np.eye(A.shape[0])
        R = (R + R.T) / 2.0
        np.fill_diagonal(R, 1.0)
        return R

    @staticmethod
    def generate_cov_matrix(size: int, mean: float, seed: int = None, std_dev: float = 0.05) -> np.ndarray:
        """
        Ritorna una matrice *di correlazione* simmetrica con 1 sulla diagonale
        e valori ~ N(mean, std_dev^2) (troncati in [-1,1]) fuori diagonale.
        Proietta a PSD ben condizionata.
        """
        if seed is not None:
            np.random.seed(seed)

        a, b = (-1 - mean) / std_dev, (1 - mean) / std_dev
        M = np.eye(size)
        for i in range(size):
            for j in range(i + 1, size):
                v = truncnorm.rvs(a, b, loc=mean, scale=std_dev)
                M[i, j] = M[j, i] = v

        M = PipelineConfronto.make_corr_well_conditioned(M, min_eig=1e-2, kappa_max=1e3)
        return M

    @staticmethod
    def make_classification_cov(n_samples: int, n_features: int, n_classes: int,
                                cov=None, shuffle: bool = True, random_state: int = None):
        """
        Dataset gaussiano multiclasse con covarianza arbitraria 'cov'.
        Centroidi definiti via codifica binaria con offset 1.0.
        """
        rng = np.random.RandomState(random_state)
        Σ = np.eye(n_features) if cov is None else np.array(cov)

        offset = 1.0
        means = []
        for k in range(n_classes):
            m = np.zeros(n_features)
            for j in range(n_features):
                if (k >> j) & 1:
                    m[j] = offset
            means.append(m)
        means = np.asarray(means)

        base, extra = divmod(n_samples, n_classes)
        X_parts, y_parts = [], []
        for cls in range(n_classes):
            n = base + (1 if cls < extra else 0)
            X_parts.append(rng.multivariate_normal(means[cls], Σ, size=n))
            y_parts.append(np.full(n, cls, dtype=int))

        X = np.vstack(X_parts)
        y = np.concatenate(y_parts)

        if shuffle:
            perm = rng.permutation(n_samples)
            X, y = X[perm], y[perm]
        return X, y

    # -------------------------- init --------------------------
    def __init__(self,
                 feature_list,
                 n_reps: int,
                 n_classes: int,
                 n_samples: int,
                 test_size: float,
                 corr_value: float,
                 out_dir: str = "output_pipeline_confront",
                 std_dev: float = 0.05,
                 random_state_split: int = 1):
        self.feature_list = list(feature_list)
        self.n_reps = int(n_reps)
        self.n_classes = int(n_classes)
        self.n_samples = int(n_samples)
        self.test_size = float(test_size)
        self.corr_value = float(corr_value)
        self.std_dev = float(std_dev)
        self.random_state_split = int(random_state_split)

        # Tuning del primo codice
        self.GB_PARAMS = dict(n_estimators=500, learning_rate=0.05, max_depth=7, random_state=1)
        self.LDAB_PARAMS = dict(n_estimators=500, learning_rate=0.05, max_depth=7, random_state=1)

        # Strutture risultati
        self.pipelines = ["LdaBoost", "GBM", "LDA+GBM"]
        self.results = {p: {f: [] for f in self.feature_list} for p in self.pipelines}

        # Output
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------- run --------------------------
    def run(self):
        for n_feat in self.feature_list:
            for rep in range(1, self.n_reps + 1):
                print(f"[features={n_feat:>3}] rep={rep}, corr={self.corr_value}")
                Σ = self.generate_cov_matrix(size=n_feat, mean=self.corr_value, seed=rep, std_dev=self.std_dev)

                X, y = self.make_classification_cov(n_samples=self.n_samples,
                                                    n_features=n_feat,
                                                    n_classes=self.n_classes,
                                                    cov=Σ,
                                                    random_state=rep)

                X_tr, X_te, y_tr, y_te = train_test_split(
                    X, y, test_size=self.test_size, random_state=self.random_state_split, stratify=y
                )

                # Pipeline 1: LdaBoost diretto
                ldaBoost = LdaBoost(**self.LDAB_PARAMS)
                ldaBoost.fit(X_tr, y_tr)
                acc_ldaBoost = accuracy_score(y_te, ldaBoost.predict(X_te))
                self.results["LdaBoost"][n_feat].append(acc_ldaBoost)

                # Pipeline 2: GradientBoosting su raw
                gb = GradientBoostingClassifier(**self.GB_PARAMS)
                gb.fit(X_tr, y_tr)
                acc_gb = accuracy_score(y_te, gb.predict(X_te))
                self.results["GBM"][n_feat].append(acc_gb)

                # Pipeline 3: LDA -> GradientBoosting
                ncomp = max(1, min(self.n_classes - 1, n_feat))
                lda = LinearDiscriminantAnalysis(n_components=ncomp)
                X_tr_lda = lda.fit_transform(X_tr, y_tr)
                X_te_lda = lda.transform(X_te)

                gb_lda = GradientBoostingClassifier(**self.GB_PARAMS)
                gb_lda.fit(X_tr_lda, y_tr)
                acc_gb_lda = accuracy_score(y_te, gb_lda.predict(X_te_lda))
                self.results["LDA+GBM"][n_feat].append(acc_gb_lda)

        return self.results

    # -------------------------- stats --------------------------
    def compute_stats(self):
        means = {p: [float(np.mean(self.results[p][f])) for f in self.feature_list] for p in self.pipelines}
        stds  = {p: [float(np.std (self.results[p][f])) for f in self.feature_list] for p in self.pipelines}
        return means, stds

    # -------------------------- save --------------------------
    def save(self):
        means, stds = self.compute_stats()

        cls_lbl = self._class_label(self.n_classes)
        n_lbl = self._samples_label(self.n_samples)

        csv_path  = self.out_dir / f"pipeline_confront_summary_{cls_lbl}_{n_lbl}.csv"
        json_sum  = self.out_dir / f"pipeline_confront_summary_{cls_lbl}_{n_lbl}.json"
        json_raw  = self.out_dir / f"pipeline_confront_raw_{cls_lbl}_{n_lbl}.json"

        # CSV summary
        with csv_path.open("w", newline="") as fh:
            writer = csv.writer(fh)
            writer.writerow(["pipeline", "correlation", "n_features", "n_reps", "n_classes", "n_samples", "mean_acc", "std_acc"])
            for p in self.pipelines:
                for idx, f in enumerate(self.feature_list):
                    writer.writerow([
                        p, self.corr_value, f, self.n_reps, self.n_classes, self.n_samples,
                        f"{means[p][idx]:.6f}", f"{stds[p][idx]:.6f}"
                    ])

        # JSON summary
        summary_json = {
            "feature_list": self.feature_list,
            "correlation": self.corr_value,
            "n_reps": self.n_reps,
            "n_classes": self.n_classes,
            "n_samples": self.n_samples,
            "means": {p: means[p] for p in self.pipelines},
            "stds":  {p: stds[p]  for p in self.pipelines},
        }
        with json_sum.open("w", encoding="utf-8") as fh:
            json.dump(summary_json, fh, ensure_ascii=False, indent=2)

        # JSON raw
        raw_json = {p: {str(f): self.results[p][f] for f in self.feature_list} for p in self.pipelines}
        with json_raw.open("w", encoding="utf-8") as fh:
            json.dump(raw_json, fh, ensure_ascii=False, indent=2)

        print(f"\nFile salvati in: {self.out_dir.resolve()}")
        print(f"- {csv_path.name}")
        print(f"- {json_sum.name}")
        print(f"- {json_raw.name}")


# Three class target, obs. 1.000

In [14]:
feature_list = [5, 50, 100, 300, 400]
n_reps       = 5
n_classes    = 3
n_samples    = 1000
test_size    = 0.30
corr_value   = 0.5

runner = PipelineConfronto(feature_list, n_reps, n_classes, n_samples, test_size, corr_value)
runner.run()
runner.save()

[features=  5] rep=1, corr=0.5
[features=  5] rep=2, corr=0.5
[features=  5] rep=3, corr=0.5
[features=  5] rep=4, corr=0.5
[features=  5] rep=5, corr=0.5
[features= 50] rep=1, corr=0.5
[features= 50] rep=2, corr=0.5
[features= 50] rep=3, corr=0.5
[features= 50] rep=4, corr=0.5
[features= 50] rep=5, corr=0.5
[features=100] rep=1, corr=0.5
[features=100] rep=2, corr=0.5
[features=100] rep=3, corr=0.5
[features=100] rep=4, corr=0.5
[features=100] rep=5, corr=0.5
[features=300] rep=1, corr=0.5
[features=300] rep=2, corr=0.5
[features=300] rep=3, corr=0.5
[features=300] rep=4, corr=0.5
[features=300] rep=5, corr=0.5
[features=400] rep=1, corr=0.5
[features=400] rep=2, corr=0.5
[features=400] rep=3, corr=0.5
[features=400] rep=4, corr=0.5
[features=400] rep=5, corr=0.5

File salvati in: /Users/giuliodonninelli/Desktop/Dati per il paper/replicazione simulazioni/output_pipeline_confront
- pipeline_confront_summary_ternary_1k.csv
- pipeline_confront_summary_ternary_1k.json
- pipeline_confront_

# Five class target, obs. 1.000

In [15]:
feature_list = [5, 50, 100, 300, 400]
n_reps       = 5
n_classes    = 5
n_samples    = 1000
test_size    = 0.30
corr_value   = 0.5

runner = PipelineConfronto(feature_list, n_reps, n_classes, n_samples, test_size, corr_value)
runner.run()
runner.save()

[features=  5] rep=1, corr=0.5
[features=  5] rep=2, corr=0.5
[features=  5] rep=3, corr=0.5
[features=  5] rep=4, corr=0.5
[features=  5] rep=5, corr=0.5
[features= 50] rep=1, corr=0.5
[features= 50] rep=2, corr=0.5
[features= 50] rep=3, corr=0.5
[features= 50] rep=4, corr=0.5
[features= 50] rep=5, corr=0.5
[features=100] rep=1, corr=0.5
[features=100] rep=2, corr=0.5
[features=100] rep=3, corr=0.5
[features=100] rep=4, corr=0.5
[features=100] rep=5, corr=0.5
[features=300] rep=1, corr=0.5
[features=300] rep=2, corr=0.5
[features=300] rep=3, corr=0.5
[features=300] rep=4, corr=0.5
[features=300] rep=5, corr=0.5
[features=400] rep=1, corr=0.5
[features=400] rep=2, corr=0.5
[features=400] rep=3, corr=0.5
[features=400] rep=4, corr=0.5
[features=400] rep=5, corr=0.5

File salvati in: /Users/giuliodonninelli/Desktop/Dati per il paper/replicazione simulazioni/output_pipeline_confront
- pipeline_confront_summary_quinary_1k.csv
- pipeline_confront_summary_quinary_1k.json
- pipeline_confront_

# Binary target, obs. 1.000

In [16]:
feature_list = [5, 50, 100, 300, 400]
n_reps       = 5
n_classes    = 2
n_samples    = 1000
test_size    = 0.30
corr_value   = 0.5

runner = PipelineConfronto(feature_list, n_reps, n_classes, n_samples, test_size, corr_value)
runner.run()
runner.save()

[features=  5] rep=1, corr=0.5
[features=  5] rep=2, corr=0.5
[features=  5] rep=3, corr=0.5
[features=  5] rep=4, corr=0.5
[features=  5] rep=5, corr=0.5
[features= 50] rep=1, corr=0.5
[features= 50] rep=2, corr=0.5
[features= 50] rep=3, corr=0.5
[features= 50] rep=4, corr=0.5
[features= 50] rep=5, corr=0.5
[features=100] rep=1, corr=0.5
[features=100] rep=2, corr=0.5
[features=100] rep=3, corr=0.5
[features=100] rep=4, corr=0.5
[features=100] rep=5, corr=0.5
[features=300] rep=1, corr=0.5
[features=300] rep=2, corr=0.5
[features=300] rep=3, corr=0.5
[features=300] rep=4, corr=0.5
[features=300] rep=5, corr=0.5
[features=400] rep=1, corr=0.5
[features=400] rep=2, corr=0.5
[features=400] rep=3, corr=0.5
[features=400] rep=4, corr=0.5
[features=400] rep=5, corr=0.5

File salvati in: /Users/giuliodonninelli/Desktop/Dati per il paper/replicazione simulazioni/output_pipeline_confront
- pipeline_confront_summary_binary_1k.csv
- pipeline_confront_summary_binary_1k.json
- pipeline_confront_ra